# Simple Spam Detector - Bidirectional LSTM Model

This notebook implements a simple binary text classifier using a
Bidirectional LSTM model to detect spam messages.

**Objective**: Achieve high precision on spam class to minimize false
positives.

## 1. Introduction & Setup

In [ ]:
"""Import required libraries and set up device configuration."""
import pickle
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from tqdm.auto import tqdm

In [ ]:
"""Set device and random seeds for reproducibility."""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set paths
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

## 2. Load Preprocessed Data

In [ ]:
"""Load train, validation, and test datasets."""
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {len(train_df):,} samples")
print(f"Val:   {len(val_df):,} samples")
print(f"Test:  {len(test_df):,} samples")
print(f"\nClass distribution (train):")
print(train_df["label"].value_counts())
print(f"\nSpam percentage: {train_df['label'].mean() * 100:.2f}%")

In [ ]:
"""Display sample messages."""
print("Sample ham message:")
print(train_df[train_df["label"] == 0].iloc[0]["text"])
print("\nSample spam message:")
print(train_df[train_df["label"] == 1].iloc[0]["text"])

## 3. Tokenization

In [ ]:
"""Build vocabulary from training data."""


def build_vocabulary(texts, min_freq=2):
    """Build word-to-index mapping from text corpus.

    Args:
        texts: List of text strings
        min_freq: Minimum word frequency to include in vocabulary

    Returns:
        Dictionary mapping words to indices
    """
    word_counts = {}
    for text in texts:
        for word in str(text).lower().split():
            word_counts[word] = word_counts.get(word, 0) + 1

    # Build vocabulary with special tokens
    vocab = {"<PAD>": 0, "<UNK>": 1}
    idx = 2
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1

    return vocab


vocab = build_vocabulary(train_df["text"])
print(f"Vocabulary size: {len(vocab):,} words")

In [ ]:
"""Calculate max sequence length based on 95th percentile."""
train_lengths = train_df["text"].apply(
    lambda x: len(str(x).split())
)
max_len = int(np.percentile(train_lengths, 95))
print(f"Max sequence length (95th percentile): {max_len}")
print(f"Mean length: {train_lengths.mean():.2f}")
print(f"Median length: {train_lengths.median():.0f}")

In [ ]:
"""Implement tokenization function."""


def tokenize(text, vocab, max_len):
    """Convert text to sequence of vocabulary indices.

    Args:
        text: Input text string
        vocab: Word-to-index mapping dictionary
        max_len: Maximum sequence length

    Returns:
        List of word indices (padded or truncated to max_len)
    """
    words = str(text).lower().split()
    indices = [
        vocab.get(word, vocab["<UNK>"]) for word in words
    ]

    # Truncate or pad
    if len(indices) > max_len:
        indices = indices[:max_len]
    else:
        indices += [vocab["<PAD>"]] * (max_len - len(indices))

    return indices


# Test tokenization
sample = train_df.iloc[0]["text"]
print(f"Original: {sample}")
print(f"Tokenized: {tokenize(sample, vocab, max_len)[:10]}...")

## 4. Create PyTorch Dataset & DataLoaders

In [ ]:
"""Define custom PyTorch Dataset class."""


class SpamDataset(Dataset):
    """PyTorch dataset for spam classification.

    Args:
        texts: List of text strings
        labels: List of binary labels (0=ham, 1=spam)
        vocab: Word-to-index mapping
        max_len: Maximum sequence length
    """

    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = self.labels.iloc[idx]

        # Tokenize
        tokens = tokenize(text, self.vocab, self.max_len)

        return {
            "tokens": torch.tensor(tokens, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.float),
        }


# Create datasets
train_dataset = SpamDataset(
    train_df["text"], train_df["label"], vocab, max_len
)
val_dataset = SpamDataset(
    val_df["text"], val_df["label"], vocab, max_len
)
test_dataset = SpamDataset(
    test_df["text"], test_df["label"], vocab, max_len
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")

In [ ]:
"""Create DataLoaders."""
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print(f"Batches per epoch: {len(train_loader)}")

## 5. Model Architecture

In [ ]:
"""Define Bidirectional LSTM model."""


class BiLSTMClassifier(nn.Module):
    """Bidirectional LSTM for binary text classification.

    Args:
        vocab_size: Size of vocabulary
        embedding_dim: Dimension of word embeddings
        hidden_dim: LSTM hidden state dimension
        dropout: Dropout probability
    """

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=64,
        dropout=0.3,
    ):
        super(BiLSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        """Forward pass.

        Args:
            x: Input token indices (batch_size, seq_len)

        Returns:
            Logits for binary classification (batch_size, 1)
        """
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)

        # Use last hidden state from both directions
        # lstm_out shape: (batch_size, seq_len, hidden_dim * 2)
        last_hidden = lstm_out[:, -1, :]

        dropped = self.dropout(last_hidden)
        logits = self.fc(dropped)

        return logits.squeeze(1)


# Initialize model
model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=64,
    dropout=0.3,
)
model = model.to(device)
print(model)

In [ ]:
"""Display model summary."""
summary(
    model,
    input_size=(BATCH_SIZE, max_len),
    dtypes=[torch.long],
    device=device,
)

## 6. Training Setup

In [ ]:
"""Calculate positive class weight for handling class imbalance."""
num_ham = (train_df["label"] == 0).sum()
num_spam = (train_df["label"] == 1).sum()
pos_weight = torch.tensor([num_ham / num_spam]).to(device)
print(f"Positive class weight: {pos_weight.item():.2f}")
print(f"(Ham: {num_ham}, Spam: {num_spam})")

In [ ]:
"""Define loss function and optimizer."""
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")

## 7. Training Loop

In [ ]:
"""Define training and evaluation functions."""


def train_epoch(model, loader, criterion, optimizer, device):
    """Train model for one epoch.

    Args:
        model: PyTorch model
        loader: Training data loader
        criterion: Loss function
        optimizer: Optimizer
        device: Device to train on

    Returns:
        Tuple of (average loss, accuracy)
    """
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Training", leave=False):
        tokens = batch["tokens"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(tokens)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model on validation/test set.

    Args:
        model: PyTorch model
        loader: Data loader
        criterion: Loss function
        device: Device to evaluate on

    Returns:
        Tuple of (average loss, accuracy)
    """
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(
            loader, desc="Evaluating", leave=False
        ):
            tokens = batch["tokens"].to(device)
            labels = batch["label"].to(device)

            outputs = model(tokens)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [ ]:
"""Train model with early stopping."""
NUM_EPOCHS = 15
PATIENCE = 3
best_val_loss = float("inf")
patience_counter = 0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Train Loss: {train_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}"
    )
    print(
        f"Val Loss: {val_loss:.4f}, " f"Val Acc: {val_acc:.4f}"
    )

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        checkpoint_path = MODEL_DIR / "lstm_baseline.pt"
        torch.save(model.state_dict(), checkpoint_path)
        print("Saved best model")
    else:
        patience_counter += 1
        print(
            f"Early stopping counter: "
            f"{patience_counter}/{PATIENCE}"
        )

    if patience_counter >= PATIENCE:
        print("Early stopping triggered")
        break

print("\nTraining complete!")

## 8. Training Visualization

In [ ]:
"""Plot training and validation curves."""
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(
    history["train_loss"], label="Train Loss", marker="o"
)
axes[0].plot(
    history["val_loss"], label="Val Loss", marker="s"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(
    history["train_acc"], label="Train Accuracy", marker="o"
)
axes[1].plot(
    history["val_acc"], label="Val Accuracy", marker="s"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training and Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Evaluation on Test Set

In [ ]:
"""Load best model and evaluate on test set."""
model.load_state_dict(
    torch.load(MODEL_DIR / "lstm_baseline.pt")
)
model.eval()

# Get predictions
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        tokens = batch["tokens"].to(device)
        labels = batch["label"].to(device)

        outputs = model(tokens)
        preds = (torch.sigmoid(outputs) > 0.5).float()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [ ]:
"""Calculate and display metrics."""
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print("Test Set Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f} (spam class)")
print(f"Recall:    {recall:.4f} (spam class)")
print(f"F1-score:  {f1:.4f} (spam class)")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["Ham", "Spam"],
        digits=4,
    )
)

In [ ]:
"""Plot confusion matrix."""
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"],
    cbar_kws={"label": "Count"},
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

print(f"\nTrue Negatives:  {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives:  {cm[1, 1]}")

In [ ]:
"""Interpret results focusing on precision."""
fp_rate = cm[0, 1] / (cm[0, 0] + cm[0, 1])
fn_rate = cm[1, 0] / (cm[1, 0] + cm[1, 1])

print("\nInterpretation:")
print(
    f"False Positive Rate: {fp_rate:.4f} "
    f"({cm[0, 1]} ham messages classified as spam)"
)
print(
    f"False Negative Rate: {fn_rate:.4f} "
    f"({cm[1, 0]} spam messages classified as ham)"
)
print(
    f"\nPrecision of {precision:.4f} means "
    f"{precision * 100:.2f}% of messages classified as spam "
    "are actually spam."
)
print(
    f"Recall of {recall:.4f} means we catch {recall * 100:.2f}% "
    "of all spam messages."
)

## 10. Save Model & Artifacts

In [ ]:
"""Save vocabulary and tokenizer configuration."""
with open(MODEL_DIR / "vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

tokenizer_config = {
    "max_len": max_len,
    "vocab_size": len(vocab),
}

with open(MODEL_DIR / "tokenizer_config.pkl", "wb") as f:
    pickle.dump(tokenizer_config, f)

print("Saved artifacts:")
print(f"- Model checkpoint: {MODEL_DIR / 'lstm_baseline.pt'}")
print(f"- Vocabulary: {MODEL_DIR / 'vocab.pkl'}")
print(
    f"- Tokenizer config: "
    f"{MODEL_DIR / 'tokenizer_config.pkl'}"
)

## 11. Summary

In [ ]:
"""Display final summary."""
print("Model Architecture:")
print("- Bidirectional LSTM")
print(f"- Embedding dimension: 128")
print(f"- Hidden dimension: 64")
print(f"- Dropout: 0.3")
print(f"- Total parameters: {sum(p.numel() for p in model.parameters()):,}")

print("\nKey Metrics:")
print(f"- Test Accuracy: {accuracy:.4f}")
print(f"- Spam Precision: {precision:.4f}")
print(f"- Spam Recall: {recall:.4f}")
print(f"- Spam F1-score: {f1:.4f}")

print("\nObservations:")
print(
    "- Model achieves good baseline performance on spam "
    "detection"
)
print(
    f"- Precision of {precision:.2f} indicates "
    f"{(1 - precision) * 100:.1f}% false positive rate"
)
print(
    f"- Recall of {recall:.2f} means {(1 - recall) * 100:.1f}% "
    "of spam messages are missed"
)

print("\nPotential Improvements:")
print("- Experiment with different embedding dimensions")
print("- Try attention mechanisms")
print("- Use pre-trained word embeddings (GloVe, Word2Vec)")
print("- Adjust decision threshold for higher precision")
print("- Add more sophisticated text preprocessing")
print("- Experiment with different architectures (CNN, Transformer)")